# 📱 Google Play Store Dataset - Data Cleaning


### 🎯 Objective
This notebook performs data cleaning and preparation on the Google Play Store dataset.

The cleaned dataset will be used for exploratory data analysis (EDA) to answer key business questions:

1. Which app categories have high download demand but low user ratings?
2. Does app size influence download volume?
3. Should the next app be free or paid to maximize business value?


### 🗂️ Steps Covered
 1. Import Libraries
 2. Load Dataset
 3. Understand dataset structure
 4. Data Cleaning
 5. Save the dataset
 
 


### 1️⃣ 📚 Import Libraries

In [2]:
# Import required libraries for data manipulation and analysis

import pandas as pd
import numpy as np

# Display all columns when exploring the dataset
pd.set_option('display.max_columns', None)

### 2️⃣ 📥Load Dataset

In [ ]:
# Load the raw Google Play Store dataset.
# The raw dataset is kept unchanged in the data/raw folder.
# All cleaning operations will be performed on this dataframe.

df = pd.read_csv("../data/raw/google_playstore_dataset.csv")

# Display the first five rows to understand the dataset structure
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,In-App Purchases,Ad Supported
0,Strike Plus,FAMILY,4.6,12460594,Varies with device,"500,000,000+",Free,0.00,Teen,Family,"January 15, 2026",Varies with device,8.0 and up,Yes,Yes
1,Tracker AI,EDUCATION,3.4,11487705,Varies with device,"500,000,000+",Free,0.00,Everyone,Education,"October 22, 2025",Varies with device,11.0 and up,Yes,Yes
2,Scanner,FAMILY,3.5,10503932,Varies with device,"500,000,000+",Paid,0.99,Everyone,Family,"November 08, 2024",Varies with device,10.0 and up,No,No
3,Connect 3D,ART_AND_DESIGN,4.7,4818860,221.6M,"100,000,000+",Free,0.00,Everyone,Art And Design,"October 07, 2025",6.8.21,10.0 and up,No,Yes
4,VPN,PRODUCTIVITY,3.5,3562800,122.9M,"100,000,000+",Free,0.00,Everyone,Productivity,"June 29, 2026",5.5.11,12.0 and up,Yes,Yes


### 3️⃣ 🔍 Understand dataset structure

In [4]:
# Check the number of rows and columns in the dataset

df.shape

(11500, 15)

In [5]:
# Display column names, data types, and missing value counts

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11500 entries, 0 to 11499
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   App               11500 non-null  str    
 1   Category          11500 non-null  str    
 2   Rating            6123 non-null   float64
 3   Reviews           11500 non-null  int64  
 4   Size              11500 non-null  str    
 5   Installs          11500 non-null  str    
 6   Type              11500 non-null  str    
 7   Price             11500 non-null  float64
 8   Content Rating    11500 non-null  str    
 9   Genres            11500 non-null  str    
 10  Last Updated      11500 non-null  str    
 11  Current Ver       11500 non-null  str    
 12  Android Ver       11500 non-null  str    
 13  In-App Purchases  11500 non-null  str    
 14  Ad Supported      11500 non-null  str    
dtypes: float64(2), int64(1), str(12)
memory usage: 1.3 MB


⭐️ **Insights:**
- The dataset has 11,500 rows and 15 columns.
- The `Rating` column contains only 6,123 non-null values, indicating that 5,377 ratings are missing. This needs to be handled before doing any analysis on ratings.
- `Size`, `Installs` are stored as text, but they actually hold numeric or date information (e.g. "19M", "500,000+", "January 15, 2026"). These need to be cleaned and converted first.
- `Reviews` and `Price` are already in the correct numeric format - no changes needed.

In [8]:
# Count the number of duplicate rows in the dataset

df.duplicated().sum()

np.int64(0)

⭐️ No duplicate rows were found in the dataset.

In [9]:
# Generate summary statistics for numerical columns
df.describe()

,Rating,Reviews,Price
count,6123.000000,1.150000e+04,11500.000000
mean,4.057211,8.550739e+03,0.217415
std,0.586312,2.045769e+05,4.169775
min,1.900000,0.000000e+00,0.000000
25%,3.600000,2.000000e+00,0.000000
50%,4.100000,5.000000e+00,0.000000
75%,4.500000,4.500000e+01,0.000000
max,5.000000,1.246059e+07,399.990000


In [11]:
# Generate summary statistics for categorical columns

df.describe(include="str")

,App,Category,Size,Installs,Type,Content Rating,Genres,Last Updated,Current Ver,Android Ver,In-App Purchases,Ad Supported
count,11500,11500,11500,11500,11500,11500,11500,11500,11500,11500,11500,11500
unique,2377,36,2821,16,2,6,79,1399,8411,10,2,2
top,Editor,GAME,Varies with device,50+,Free,Everyone,Family,"May 15, 2026",Varies with device,10.0 and up,Yes,Yes
freq,251,2061,1778,2096,11027,7899,1388,33,1778,2855,7693,8866


⭐️ **Insights:**
- The dataset contains 36 unique app `Category`, with GAME being the most common.
- The `Size` column contains many unique values, and "Varies with device" is the most frequent value.
- The `Installs` column has 16 unique values and will need to be converted to a numeric format.
- Most applications are Free `Type`.
- `In-App Purchases` and `Ad Supported` are binary features (Yes/No) and may be useful for further analysis.

In [14]:
# Display all unique values in the Category column
# This helps identify the different formats and determine the required cleaning steps.
df["Category"].unique()

<StringArray>
[                 'FAMILY',               'EDUCATION',
          'ART_AND_DESIGN',            'PRODUCTIVITY',
 'ARTIFICIAL_INTELLIGENCE',                 'MEDICAL',
                   'TOOLS',                  'SOCIAL',
                    'GAME',                 'WEATHER',
             'PHOTOGRAPHY',     'MAPS_AND_NAVIGATION',
         'PERSONALIZATION',      'LIBRARIES_AND_DEMO',
          'FOOD_AND_DRINK',               'PARENTING',
                 'FINANCE',           'VIDEO_PLAYERS',
               'LIFESTYLE',          'HOUSE_AND_HOME',
                'BUSINESS',                  'EVENTS',
           'COMMUNICATION',         'WEB3_AND_CRYPTO',
     'BOOKS_AND_REFERENCE',        'TRAVEL_AND_LOCAL',
                'SHOPPING',      'NEWS_AND_MAGAZINES',
           'ENTERTAINMENT',                  'COMICS',
         'VIRTUAL_REALITY',       'AUTO_AND_VEHICLES',
                  'SPORTS',                  'BEAUTY',
                  'DATING',      'HEALTH_AND_FITNES

⭐️ **Insights**:
* The `Category` column contains 36 unique app categories.
* No missing, invalid, or inconsistent category values were identified.No data cleaning is required for this column.

In [19]:
# Display all unique values in the Size column.
# This helps identify the different size formats and determine the required cleaning steps.

df["Size"].unique()

<StringArray>
['Varies with device',             '221.6M',             '122.9M',
             '187.4M',              '15.6M',              '80.1M',
             '134.0M',             '127.5M',             '321.5M',
              '52.9M',
 ...
            '1830.4M',             '247.1M',             '183.1M',
            '1448.4M',              '76.7M',             '287.8M',
             '896.4M',             '625.0M',            '1087.9M',
             '219.9M']
Length: 2821, dtype: str

In [25]:
# Check if any size values are stored in kilobytes
df[df["Size"].str.contains("k", case=False, na=False)]["Size"].unique()

<StringArray>
[]
Length: 0, dtype: str

⭐️ **Insights**:
* The `Size` column is stored as a string.
* All numeric values are represented in megabytes (M).
* The column contains "Varies with device", which is not a numeric value.
* The column will be converted to a numeric format (MB) for analysis.

In [15]:
# Display all unique values in the Installs column.
# This helps identify the different formats and determine the required cleaning steps.

df["Installs"].unique()

<StringArray>
['500,000,000+', '100,000,000+',  '50,000,000+',  '10,000,000+',
   '5,000,000+',   '1,000,000+',     '500,000+',     '100,000+',
      '50,000+',      '10,000+',       '5,000+',       '1,000+',
         '500+',         '100+',          '50+',          '10+']
Length: 16, dtype: str

⭐️ **Insights**:
* The `Installs` column is stored as a string. Values contain commas (,) and a plus sign (+).
* No invalid or unexpected values were found.
* The column will be converted to a numeric data type for analysis.

In [26]:
# Display all unique values in the Type column.
# This helps identify the different app types and detect any unexpected values.

df["Type"].unique()

<StringArray>
['Free', 'Paid']
Length: 2, dtype: str

⭐️ **Insights**:
* The `Type` column contains only Free and Paid values.
* No missing or inconsistent values were identified. No data cleaning is required for this column.

### 4️⃣ 🧹 Data Cleaning

To prepare the dataset for exploratory data analysis (EDA), the following data cleaning tasks will be performed:
* Handle missing values in the `Rating` column.
* Convert the `Size` column to a numeric format.
* Convert the `Installs` column to a numeric format.

##### 🔷 Handle missing values in the `Rating` column.

In [33]:
# Count the number of missing values in the Rating column
df["Rating"].isnull().sum()

np.int64(5377)

In [30]:
df[df["Rating"].isnull()][["Rating","App", "Reviews", "Installs"]].sample(10)

,Rating,App,Reviews,Installs
10996,NaN,VPN,0,10+
7872,NaN,Browser 2025,3,50+
5868,NaN,Smart Tracker,4,100+
7128,NaN,Pro Pay 2026,0,100+
9835,NaN,Hyper Browser Pro 2025,4,10+
5763,NaN,Smart Planner,4,100+
10069,NaN,Ultra Fit,3,10+
7308,NaN,Chat,0,100+
7142,NaN,Crypto Learn Max,0,100+
6108,NaN,Daily Maker 2025,3,100+


✅️ **Result**:

The `Rating` column contains 5,377 missing values.

A sample of the missing records shows that many of these apps have very few or no reviews, which likely explains the absence of ratings.<br>
The missing values will be retained as NaN to preserve the dataset and avoid introducing artificial ratings.

##### 🔷 Convert the `Size` column to a numeric format.

In [36]:
# Replace 'Varies with device' with NaN since it is not a numeric size.

df["Size"] = df["Size"].replace("Varies with device", np.nan)

In [38]:
# Remove the 'M' suffix from app sizes.
# Convert the cleaned values to a numeric data type (float).

df["Size"] = (
    df["Size"]
      .str.replace("M", "", regex=False)
      .astype(float))

In [39]:
# Verify the data type and preview the cleaned values.

print(df["Size"].dtype)
df["Size"].head()

float64


0      NaN
1      NaN
2      NaN
3    221.6
4    122.9
Name: Size, dtype: float64

✅ **Result**:
* The `Size` column was successfully converted from a string to a numeric (float) data type.
* The "M" suffix was removed from all size values.
* "Varies with device" values were replaced with NaN.

##### 🔷 Convert the `Installs` column to a numeric format.

In [34]:
# Remove commas (,) and the plus sign (+) from the Installs column.
# Convert the cleaned values from string to integer for numerical analysis.
df["Installs"] = (
    df["Installs"]
      .str.replace(",", "", regex=False)
      .str.replace("+", "", regex=False)
      .astype(int)
)

In [35]:
# Verify the data type and preview the cleaned values.

print(df["Installs"].dtype)
df["Installs"].head()

int64


0    500000000
1    500000000
2    500000000
3    100000000
4    100000000
Name: Installs, dtype: int64

✅ **Result**:
- The `Installs` column was successfully converted from a string to an integer data type.
- Commas and the plus sign were removed from the installation values.

### 5️⃣ 💾 Save Cleaned Dataset

In [40]:
# Save the cleaned dataset for use in the EDA notebook.
df.to_csv("../data/cleaned/googleplaystore_cleaned.csv", index=False)
print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
